In [1]:
import pandas as pd

In [44]:
# 1. Setup Initial Data
stock_df = pd.DataFrame({
    'Stock1': [0.03],
    'Stock2': [0.025],
    'Stock3': [0.018],
    'Stock4': [0.006],
    'Stock5': [0.002],
    'Stock6': [-0.009],
    'Stock7': [-0.017],
    'Stock8': [-0.028],
}, index=['1-Feb'])
stock_df

,Stock1,Stock2,Stock3,Stock4,Stock5,Stock6,Stock7,Stock8
1-Feb,0.03,0.025,0.018,0.006,0.002,-0.009,-0.017,-0.028


In [45]:
# 2. Alpha value: Rank (-returns) scaled 0 to 1
# rank(axis=1) ranks the columns within the row
alpha = stock_df.rank(axis=1, ascending=False)
alpha.rename({'1-Feb': '2-Feb'}, inplace=True)
min_val = alpha.min(axis=1).values[0]
max_val = alpha.max(axis=1).values[0]
alpha_scaled = (alpha - min_val) / (max_val - min_val)
alpha_scaled

,Stock1,Stock2,Stock3,Stock4,Stock5,Stock6,Stock7,Stock8
2-Feb,0.0,0.142857,0.285714,0.428571,0.571429,0.714286,0.857143,1.0


In [46]:
# 3. Centred around 0: Subtract the row mean (0.50), Sum of all values = 0
centred = alpha_scaled.sub(alpha_scaled.mean(axis=1), axis=0).round(2)
centred

,Stock1,Stock2,Stock3,Stock4,Stock5,Stock6,Stock7,Stock8
2-Feb,-0.5,-0.36,-0.21,-0.07,0.07,0.21,0.36,0.5


In [48]:
# 4. Normalized Weights: Divide by the sum of absolute values
# By normalize, we mean that the total absolute sum of weights is 1.
total_abs_weight = centred.abs().sum(axis=1)
print(f'Sum of the absolute values: {total_abs_weight.round(1)}')
norm_weights = centred.div(total_abs_weight, axis=0).round(2)
norm_weights

Sum of the absolute values: 2-Feb    2.3
dtype: float64


,Stock1,Stock2,Stock3,Stock4,Stock5,Stock6,Stock7,Stock8
2-Feb,-0.22,-0.16,-0.09,-0.03,0.03,0.09,0.16,0.22


In [49]:
# 5. Allocate Capital: $20 Million Position
capital = 20
positions_mn = norm_weights * capital
positions_mn

,Stock1,Stock2,Stock3,Stock4,Stock5,Stock6,Stock7,Stock8
2-Feb,-4.4,-3.2,-1.8,-0.6,0.6,1.8,3.2,4.4


In [50]:
# 6. Multiply positions by returns to get P&L per stock
actual_returns = [0.02, 0.02, -0.021, -0.021, 0.01, 0.0, 0.01, 0.02]
profit_mn = positions_mn * actual_returns
profit_mn.round(2)

,Stock1,Stock2,Stock3,Stock4,Stock5,Stock6,Stock7,Stock8
2-Feb,-0.09,-0.06,0.04,0.01,0.01,0.0,0.03,0.09


In [51]:
# 7. Calculate the total Daily P&L
total_daily_pnl = profit_mn.sum(axis=1)
total_daily_pnl

2-Feb    0.0244
dtype: float64

# Decay Example

In [52]:
decay_df = pd.DataFrame({
    'Stock1': [0.03, -0.03, -0.22],
    'Stock2': [-0.03, -0.16, -0.16],
    'Stock3': [0.09, -0.22, -0.09],
    'Stock4': [-0.16, 0.03, -0.03],
    'Stock5': [0.22, -0.09, 0.03],
    'Stock6': [-0.22, 0.22, 0.09],
    'Stock7': [0.16, 0.09, 0.16],
    'Stock8': [-0.09, 0.16, 0.22],
}, index=['31-Jan', '1-Feb', '2-Feb'])
decay_df

,Stock1,Stock2,Stock3,Stock4,Stock5,Stock6,Stock7,Stock8
31-Jan,0.03,-0.03,0.09,-0.16,0.22,-0.22,0.16,-0.09
1-Feb,-0.03,-0.16,-0.22,0.03,-0.09,0.22,0.09,0.16
2-Feb,-0.22,-0.16,-0.09,-0.03,0.03,0.09,0.16,0.22


In [58]:
# decay_linear(x, n)
final_norm_weights = (decay_df.iloc[2] * 3 + decay_df.iloc[1] * 2 + decay_df.iloc[0] * 1) / (3 + 2 + 1)
decay_df.loc['Final weight'] = final_norm_weights.round(2)
decay_df

,Stock1,Stock2,Stock3,Stock4,Stock5,Stock6,Stock7,Stock8
31-Jan,0.03,-0.03,0.09,-0.16,0.22,-0.22,0.16,-0.09
1-Feb,-0.03,-0.16,-0.22,0.03,-0.09,0.22,0.09,0.16
2-Feb,-0.22,-0.16,-0.09,-0.03,0.03,0.09,0.16,0.22
Final weight,-0.12,-0.14,-0.10,-0.03,0.02,0.08,0.14,0.15
